# deep_cap_failure_attribution_v1

Notebook-first diagnostic for the latest `deep_capitulation_reentry_guard_v1` result.

Purpose:
- Do not tune another path threshold.
- Add the latest deep-cap candidate to the existing OOS poor-window and feature-delta aggregate.
- Classify whether the next model loop should target participation, drawdown control, or selection bias.
- Keep the conclusion inside this runtime notebook; sidecar JSON/CSV files are machine-readable evidence only.

## External Basis

- Qlib paper/GitHub: keep the experiment inside a reproducible AI quant workflow with persisted data, learning, backtest, and deployment artifacts.
- Bailey, Borwein, Lopez de Prado, Zhu PBO/CSCV: repeated strategy trials need out-of-sample and walk-forward discipline before promotion.
- Focal loss and class-balanced loss papers: previous rebound-weighted and state-conditioned-loss screens already tested imbalance-aware learning; this notebook uses the failure to route the next mechanism instead of repeating the same weighting grid.

In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "AGENTS.md").exists() and (path / "src/soxl_quant").exists():
            return path
    raise RuntimeError("could not locate repository root")

ROOT = find_repo_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.oos_diagnostics import (
    aggregate_feature_differences,
    aggregate_poor_windows,
    attribute_gate_gaps,
    normalise_gate_metrics_from_selection,
    render_gate_gap_review,
    run_oos_diagnostic,
    summarize_gate_gap_attribution,
)

CARD = ROOT / "runtime/model_selection/card/deep_cap_failure_attribution_v1"
DIAG_DIR = CARD / "diagnostics/deep_capitulation_reentry_guard_v1"
AGG_DIR = CARD / "aggregate"
PREV_AGG = ROOT / "runtime/model_selection/card/oos_poor_window_factor_diff_aggregate_v1/aggregate"
DEEP_DAILY = ROOT / "runtime/execution/card/deep_capitulation_reentry_guard_v1/selected_variant_daily.csv"
DEEP_SELECTION = ROOT / "runtime/execution/card/deep_capitulation_reentry_guard_v1/selection.json"

TARGET_EXCESS = 30.0
TARGET_MAX_DD = 15.0

EXTERNAL_REFERENCES = [
    {
        "label": "Qlib: An AI-oriented Quantitative Investment Platform",
        "url": "https://arxiv.org/abs/2009.11189",
        "note": "research-to-production workflow and persisted quant artifacts",
    },
    {
        "label": "Probability of Backtest Overfitting",
        "url": "https://doi.org/10.21314/JCF.2016.322",
        "note": "use OOS/walk-forward evidence before promoting repeated trials",
    },
    {
        "label": "Focal Loss",
        "url": "https://arxiv.org/abs/1708.02002",
        "note": "imbalance-aware learning reference already tested in prior rebound-loss screens",
    },
    {
        "label": "Class-Balanced Loss",
        "url": "https://arxiv.org/abs/1901.05555",
        "note": "effective-number reweighting reference; do not repeat if failure is not weight-only",
    },
]

CARD.mkdir(parents=True, exist_ok=True)
DIAG_DIR.mkdir(parents=True, exist_ok=True)
AGG_DIR.mkdir(parents=True, exist_ok=True)
print(CARD.relative_to(ROOT))

runtime/model_selection/card/deep_cap_failure_attribution_v1


## 1. Baseline and Non-Duplication

This run compares against the existing aggregate `oos_poor_window_factor_diff_aggregate_v1` and the latest `deep_capitulation_reentry_guard_v1` execution screen.

Closest success case: stable rebound/deep-cap labels identified strong forward-return lift, and the deep-cap execution guard improved strict OOS return.

Closest failure case: state-head/fold-consensus removed unstable rebound target lifts, and deep-cap execution still failed the hard MaxDD target.

Why this is not a duplicate: this notebook does not introduce another label, threshold, or cap. It performs failure attribution after the latest candidate so the next training objective can be routed with evidence.

In [2]:
deep_summary = json.loads((ROOT / "runtime/execution/card/deep_capitulation_reentry_guard_v1/summary.json").read_text())
deep_selection = json.loads(DEEP_SELECTION.read_text())
prev_manifest = json.loads((PREV_AGG / "manifest.json").read_text())

print(json.dumps({
    "deep_decision": deep_summary["decision"],
    "deep_selected": deep_summary["selected_label"],
    "deep_current_return": round(deep_summary["selected_current_return"], 3),
    "deep_current_relative_return": round(deep_summary["selected_current_relative_return"], 3),
    "deep_current_max_dd": round(deep_summary["selected_current_max_dd"], 3),
    "deep_worst_poor_window": deep_summary["worst_poor_window"],
    "previous_aggregate_profiles": prev_manifest["labels"],
}, indent=2))

{
  "deep_decision": "diagnostic_keep_no_promotion",
  "deep_selected": "deep_cap_m0d30_x0d35_thr0d02_lock3",
  "deep_current_return": 3718.598,
  "deep_current_relative_return": 2902.085,
  "deep_current_max_dd": 32.809,
  "deep_worst_poor_window": {
    "window_start": "2026-02-09",
    "window_end": "2026-05-11",
    "strategy_return": 2.781625465989901,
    "buyhold_return": 197.3918503814099,
    "excess_pp": -194.61022491542002,
    "strategy_max_dd": 5.901437000520582,
    "avg_position": 0.13266223093750001
  },
  "previous_aggregate_profiles": [
    "current_runtime_paper",
    "balanced_rule_residual_v19_reg",
    "balanced_rule_residual_v22",
    "drawdown_rebound_reward_train_v1",
    "mode_conditioned_rebound_reward_train_v2"
  ]
}


## 2. Generate Deep-Cap OOS Diagnostics

`run_oos_diagnostic` writes poor windows and feature deltas for the selected deep-cap daily path. The diagnostic is intentionally not promotion evidence; it is a routing artifact for the next training/model-selection loop.

In [3]:
manifest = run_oos_diagnostic(
    daily_csv=DEEP_DAILY,
    profile="deep_capitulation_reentry_guard_v1",
    output_dir=DIAG_DIR,
    oos_start="2020-01-02",
    window_size=63,
    top_windows=20,
    top_features=40,
    target_excess=TARGET_EXCESS,
    target_max_dd=TARGET_MAX_DD,
    external_references=EXTERNAL_REFERENCES,
)

poor_windows = pd.read_csv(DIAG_DIR / "poor_windows.csv")
feature_differences = pd.read_csv(DIAG_DIR / "feature_differences.csv")

print(json.dumps(manifest, indent=2))
display(poor_windows.head(8))
display(feature_differences.head(10))

[44446:MainThread](2026-05-26 00:05:32,227) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[44446:MainThread](2026-05-26 00:05:33,094) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[44446:MainThread](2026-05-26 00:05:33,095) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[44446:MainThread](2026-05-26 00:05:35,629) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | Loading data Done


[44446:MainThread](2026-05-26 00:05:35,631) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[44446:MainThread](2026-05-26 00:05:35,634) INFO - qlib.timer - [log.py:127] - Time cost: 0.007s | Init data Done


{
  "profile": "deep_capitulation_reentry_guard_v1",
  "daily_csv": "/Users/rziiiii/.openclaw/workspace/runtime/execution/card/deep_capitulation_reentry_guard_v1/selected_variant_daily.csv",
  "oos_start": "2020-01-02",
  "oos_end": "2026-05-21",
  "window_size": 63,
  "target_excess": 30.0,
  "target_max_dd": 15.0,
  "artifacts": {
    "windows": "/Users/rziiiii/.openclaw/workspace/runtime/model_selection/card/deep_cap_failure_attribution_v1/diagnostics/deep_capitulation_reentry_guard_v1/poor_windows.csv",
    "features": "/Users/rziiiii/.openclaw/workspace/runtime/model_selection/card/deep_cap_failure_attribution_v1/diagnostics/deep_capitulation_reentry_guard_v1/feature_differences.csv",
    "review": "/Users/rziiiii/.openclaw/workspace/runtime/model_selection/card/deep_cap_failure_attribution_v1/diagnostics/deep_capitulation_reentry_guard_v1/oos_diagnostic_review.md"
  },
  "external_references": [
    {
      "label": "Qlib: An AI-oriented Quantitative Investment Platform",
      "

,start,end,rows,strategy_return,buyhold_return,excess_return,max_dd,avg_pos,bad_score
0,2026-02-10,2026-05-11,63,3.688139,200.631509,-196.943370,5.901437,0.121890,202.844807
1,2026-02-05,2026-05-06,63,19.569758,211.455411,-191.885653,5.901437,0.160523,197.787090
2,2026-02-12,2026-05-13,63,3.921546,191.980998,-188.059452,4.536602,0.096135,192.596054
3,2026-02-13,2026-05-14,63,1.968102,188.264457,-186.296355,4.536602,0.083257,190.832957
4,2026-02-09,2026-05-08,63,2.781625,176.339225,-173.557600,5.901437,0.134768,179.459037
5,2026-02-23,2026-05-21,63,-0.235147,170.862432,-171.097579,4.536602,0.023810,175.634181
6,2025-04-08,2025-07-09,63,70.229491,223.030298,-152.800808,12.174846,0.504842,164.975654
7,2026-02-20,2026-05-20,63,-1.164264,158.083736,-159.248000,4.536602,0.031746,163.784602


,feature,bad_mean,reference_mean,delta,delta_z,bad_coverage,reference_coverage,spearman_to_excess,abs_delta_z
0,jump_variation_5,1.246131,0.107758,1.138373,1.422645,1.0,1.0,0.005931,1.422645
1,vol_of_vol_21,0.224059,0.116356,0.107703,1.167749,1.0,1.0,-0.022348,1.167749
2,realized_vol_63,1.367109,1.014883,0.352226,1.078544,1.0,1.0,-0.008581,1.078544
3,realized_kurtosis_21,1.531505,0.266375,1.265131,0.946721,1.0,1.0,-0.040486,0.946721
4,downside_ratio_21,0.405241,0.470761,-0.065520,-0.766873,1.0,1.0,0.191253,0.766873
5,mom_21,0.260277,0.061823,0.198454,0.763083,1.0,1.0,-0.188716,0.763083
6,expectancy_proxy_21,0.208977,0.067903,0.141074,0.758071,1.0,1.0,-0.212414,0.758071
7,relative_strength_21,1.699478,0.386365,1.313114,0.752444,1.0,1.0,-0.187270,0.752444
8,trend_slope_21,0.098747,0.013918,0.084829,0.739347,1.0,1.0,-0.009829,0.739347
9,vix_percentile_252,0.687653,0.452825,0.234828,0.730257,1.0,1.0,0.113313,0.730257


## 3. Aggregate With Existing Poor-Window Evidence

The previous aggregate covered current runtime paper, v19/v22 actor paths, and two rebound reward training branches. This step adds the deep-cap guard as a sixth profile and recomputes shared failure clusters and stable feature deltas.

In [4]:
input_dirs = [Path(path) for path in prev_manifest["input_dirs"]] + [DIAG_DIR]
labels = list(prev_manifest["labels"]) + ["deep_capitulation_reentry_guard_v1"]

def read_csv_if_exists(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

windows_by_profile = {label: read_csv_if_exists(directory / "poor_windows.csv") for label, directory in zip(labels, input_dirs)}
features_by_profile = {label: read_csv_if_exists(directory / "feature_differences.csv") for label, directory in zip(labels, input_dirs)}

clusters = aggregate_poor_windows(
    windows_by_profile,
    target_excess=TARGET_EXCESS,
    target_max_dd=TARGET_MAX_DD,
)
feature_clusters = aggregate_feature_differences(features_by_profile, top_n=40)

clusters.to_csv(AGG_DIR / "poor_window_clusters.csv", index=False)
feature_clusters.to_csv(AGG_DIR / "feature_delta_clusters.csv", index=False)

display(clusters)
display(feature_clusters.head(15))

,cluster_id,cluster_start,cluster_end,profile_count,window_count,profiles,mean_excess_return,worst_excess_return,mean_max_dd,max_max_dd,mean_avg_pos,mean_bad_score,max_bad_score,dominant_failure_type,primary_optimization_path
0,C002,2025-04-04,2025-07-22,6,37,"balanced_rule_residual_v19_reg,balanced_rule_r...",-155.465708,-199.393437,7.106952,16.324924,0.522852,162.572660,203.420490,underparticipation,reentry_participation_or_behavior_clone
1,C003,2026-02-04,2026-05-21,5,31,"balanced_rule_residual_v22,current_runtime_pap...",-165.796419,-198.889223,8.462501,24.118142,0.092394,174.258919,205.310696,underparticipation,reentry_participation_or_behavior_clone
2,C001,2020-03-18,2020-06-19,4,12,"balanced_rule_residual_v19_reg,balanced_rule_r...",-158.096497,-174.787005,8.683677,13.885509,0.514949,166.780175,182.311302,underparticipation,reentry_participation_or_behavior_clone


,feature,profile_count,mean_delta_z,mean_abs_delta_z,direction_consistency,mean_spearman_to_excess,profiles,stability_score
0,vol_of_vol_21,6,2.618836,2.618836,1.0,-0.028262,"balanced_rule_residual_v19_reg,balanced_rule_r...",5.096019
1,tracking_error_vol_21,4,3.153269,3.153269,1.0,0.008294,"balanced_rule_residual_v19_reg,balanced_rule_r...",5.074991
2,decay_63,6,-2.423424,2.423424,1.0,0.065161,"balanced_rule_residual_v19_reg,balanced_rule_r...",4.715765
3,realized_vol_63,6,2.286689,2.286689,1.0,-0.014036,"balanced_rule_residual_v19_reg,balanced_rule_r...",4.449692
4,tracking_error_21,5,-1.765068,1.765068,1.0,0.090525,"balanced_rule_residual_v19_reg,balanced_rule_r...",3.162577
5,realized_vol_21,6,1.469823,1.469823,1.0,-0.002399,"balanced_rule_residual_v19_reg,balanced_rule_r...",2.860143
6,vol_coef_21,6,1.273677,1.273677,1.0,-0.003044,"balanced_rule_residual_v19_reg,balanced_rule_r...",2.478462
7,jump_variation_5,6,1.175330,1.175330,1.0,0.005292,"balanced_rule_residual_v19_reg,balanced_rule_r...",2.287086
8,vix_level,4,1.270719,1.270719,1.0,0.097231,"balanced_rule_residual_v19_reg,balanced_rule_r...",2.045143
9,realized_vol_10,6,1.015159,1.015159,1.0,-0.000513,"balanced_rule_residual_v19_reg,balanced_rule_r...",1.975407


## 4. Gate-Gap Attribution For Deep-Cap Candidate

The rolling poor-window view says where the path failed locally. The gate-gap view says why the candidate still fails the hard objective: `relative_return >= +30pp` and `MaxDD <= 15%` on strict OOS plus walk-forward windows.

In [5]:
def deep_cap_gate_metrics(selection: dict) -> dict[str, dict[str, float]]:
    out: dict[str, dict[str, float]] = {}
    current = dict(selection.get("current_metrics", {}))
    if current:
        out["strict_oos"] = {
            "return": current.get("current_return"),
            "buy_and_hold_return": current.get("current_buy_and_hold_return"),
            "relative_return": current.get("current_relative_return"),
            "sharpe": current.get("current_sharpe"),
            "max_dd": current.get("current_max_dd"),
            "avg_pos": current.get("current_avg_pos"),
        }
    for row in selection.get("walkforward_metrics", []):
        label = str(row.get("wf_window"))
        out[label] = {
            "return": row.get("wf_return"),
            "buy_and_hold_return": row.get("wf_buy_and_hold_return"),
            "relative_return": row.get("wf_relative_return"),
            "sharpe": row.get("wf_sharpe"),
            "max_dd": row.get("wf_max_dd"),
            "avg_pos": row.get("wf_avg_pos"),
        }
    return out

metrics_by_profile = {
    "deep_capitulation_reentry_guard_v1": deep_cap_gate_metrics(deep_selection),
}

gate_gaps = attribute_gate_gaps(
    metrics_by_profile,
    target_relative_return=TARGET_EXCESS,
    target_max_dd=TARGET_MAX_DD,
)
gate_summary = summarize_gate_gap_attribution(gate_gaps)

gate_gaps.to_csv(AGG_DIR / "gate_gap_attribution.csv", index=False)
gate_summary.to_csv(AGG_DIR / "gate_gap_profile_summary.csv", index=False)

display(gate_summary)
display(gate_gaps)

,profile,window_count,gate_pass_count,failure_count,dominant_failure_type,worst_window,max_severity_score,recommended_next_mechanism
0,deep_capitulation_reentry_guard_v1,4,0,4,drawdown_control,wf_2022,217.468289,drawdown_controller_or_sequence_risk_feature


,profile,window,return,buy_and_hold_return,relative_return,sharpe,max_dd,avg_pos,relative_gap,max_dd_gap,relative_pass,maxdd_pass,gate_pass,failure_type,severity_score,primary_optimization_path
0,deep_capitulation_reentry_guard_v1,wf_2022,-32.994141,-86.588072,53.593931,-1.341123,36.746829,0.170474,0.0,21.746829,True,False,False,drawdown_control,217.468289,drawdown_controller_or_sequence_risk_feature
1,deep_capitulation_reentry_guard_v1,wf_2023_2024,235.111654,198.504290,36.607364,1.533543,33.780768,0.485162,0.0,18.780768,True,False,False,drawdown_control,187.807685,drawdown_controller_or_sequence_risk_feature
2,deep_capitulation_reentry_guard_v1,strict_oos,3718.597739,816.512500,2902.085239,1.394646,32.809415,0.457935,0.0,17.809415,True,False,False,drawdown_control,178.094152,drawdown_controller_or_sequence_risk_feature
3,deep_capitulation_reentry_guard_v1,wf_2020,96.588263,58.093571,38.494692,1.381838,26.256455,0.419786,0.0,11.256455,True,False,False,drawdown_control,112.564549,drawdown_controller_or_sequence_risk_feature


## 5. Decision Handoff

This cell writes the machine-readable conclusion. The notebook itself remains the primary human-readable runtime conclusion.

In [6]:
def frame_head_records(frame: pd.DataFrame, n: int = 5) -> list[dict]:
    if frame.empty:
        return []
    return json.loads(frame.head(n).to_json(orient="records"))

def render_aggregate_review(clusters: pd.DataFrame, features: pd.DataFrame, gate_gaps: pd.DataFrame, gate_summary: pd.DataFrame) -> str:
    lines = [
        "# Deep-Cap Failure Attribution",
        "",
        "## External References",
        "",
    ]
    for ref in EXTERNAL_REFERENCES:
        lines.append(f"- {ref['label']}: {ref['url']} — {ref['note']}")
    lines.extend([
        "",
        "## Shared Poor-Window Clusters After Adding Deep-Cap",
        "",
        "| cluster | start | end | profiles | type | mean excess | max dd | avg pos | path |",
        "| --- | --- | --- | ---: | --- | ---: | ---: | ---: | --- |",
    ])
    for row in clusters.itertuples(index=False):
        lines.append(
            f"| {row.cluster_id} | {row.cluster_start} | {row.cluster_end} | {int(row.profile_count)} | "
            f"{row.dominant_failure_type} | {float(row.mean_excess_return):.2f}pp | "
            f"{float(row.max_max_dd):.2f}% | {float(row.mean_avg_pos):.3f} | {row.primary_optimization_path} |"
        )
    lines.extend([
        "",
        "## Top Stable Feature Deltas",
        "",
        "| feature | profiles | mean delta_z | consistency | score |",
        "| --- | ---: | ---: | ---: | ---: |",
    ])
    for row in features.head(12).itertuples(index=False):
        lines.append(
            f"| {row.feature} | {int(row.profile_count)} | {float(row.mean_delta_z):.3f} | "
            f"{float(row.direction_consistency):.3f} | {float(row.stability_score):.3f} |"
        )
    lines.extend(["", render_gate_gap_review(gate_gaps, gate_summary).strip(), ""])
    return "\n".join(lines)

review = render_aggregate_review(clusters, feature_clusters, gate_gaps, gate_summary)
(AGG_DIR / "aggregate_oos_review.md").write_text(review, encoding="utf-8")

top_cluster = clusters.iloc[0].to_dict() if not clusters.empty else {}
top_features = frame_head_records(feature_clusters, 12)
worst_gate = gate_gaps.iloc[0].to_dict() if not gate_gaps.empty else {}

underparticipation_clusters = int((clusters.get("dominant_failure_type", pd.Series(dtype=str)) == "underparticipation").sum()) if not clusters.empty else 0
drawdown_or_joint_gate_rows = int(gate_gaps[gate_gaps["failure_type"].isin(["drawdown_control", "joint_tradeoff", "near_miss_drawdown"])].shape[0]) if not gate_gaps.empty else 0

if drawdown_or_joint_gate_rows and underparticipation_clusters:
    decision = "diagnostic_handoff_joint_participation_drawdown_no_promotion"
    next_mechanism = "state_conditioned_participation_objective_with_explicit_drawdown_controller"
elif underparticipation_clusters:
    decision = "diagnostic_handoff_underparticipation_no_promotion"
    next_mechanism = "participation_or_reentry_objective"
else:
    decision = "diagnostic_handoff_drawdown_or_selection_no_promotion"
    next_mechanism = "walk_forward_weighted_multi_objective_selection"

selection = {
    "experiment": "deep_cap_failure_attribution_v1",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "source_experiment": "deep_capitulation_reentry_guard_v1",
    "direct_comparison_baseline": "oos_poor_window_factor_diff_aggregate_v1",
    "decision": decision,
    "next_mechanism": next_mechanism,
    "promotion_allowed": False,
    "full_gate": False,
    "promotion_blocker": "This is a diagnostic attribution notebook; deep-cap still fails the <=15% MaxDD objective and walk-forward gate.",
    "target_excess_pp": TARGET_EXCESS,
    "target_max_dd_pct": TARGET_MAX_DD,
    "cluster_count": int(len(clusters)),
    "top_cluster": top_cluster,
    "deep_cap_gate_summary": frame_head_records(gate_summary, 5),
    "worst_gate_gap": worst_gate,
    "top_feature_deltas": top_features,
    "external_sources": [ref["url"] for ref in EXTERNAL_REFERENCES],
    "sidecars": [
        "diagnostics/deep_capitulation_reentry_guard_v1/poor_windows.csv",
        "diagnostics/deep_capitulation_reentry_guard_v1/feature_differences.csv",
        "diagnostics/deep_capitulation_reentry_guard_v1/oos_diagnostic_review.md",
        "aggregate/poor_window_clusters.csv",
        "aggregate/feature_delta_clusters.csv",
        "aggregate/gate_gap_attribution.csv",
        "aggregate/gate_gap_profile_summary.csv",
        "aggregate/aggregate_oos_review.md",
        "summary.json",
    ],
    "acceptance_gate_for_next_run": {
        "relative_return_min_pp": TARGET_EXCESS,
        "max_dd_max_pct": TARGET_MAX_DD,
        "fixed_split_required": True,
        "walkforward_required": True,
        "avoid_more_deep_cap_threshold_grids": True,
        "next_training_must_jointly_address_participation_and_drawdown": bool(drawdown_or_joint_gate_rows and underparticipation_clusters),
    },
}

summary = {
    "experiment": selection["experiment"],
    "decision": decision,
    "source_deep_cap_decision": deep_summary["decision"],
    "source_deep_cap_selected": deep_summary["selected_label"],
    "deep_cap_current_relative_return": deep_summary["selected_current_relative_return"],
    "deep_cap_current_max_dd": deep_summary["selected_current_max_dd"],
    "cluster_count_after_deep_cap": int(len(clusters)),
    "top_cluster": top_cluster,
    "top_features": top_features,
    "gate_summary": selection["deep_cap_gate_summary"],
    "conclusion": [
        "Deep-cap improves strict OOS return but does not solve the hard <=15% MaxDD objective.",
        "Adding deep-cap to the previous aggregate keeps the shared bad-window route in underparticipation clusters, especially 2025-04..2025-07 and 2026-02..2026-05.",
        "The gate-gap view shows MaxDD/joint trade-off remains the blocker; another participation-only threshold grid is not justified.",
        "Next notebook should screen a state-conditioned participation objective with an explicit drawdown controller or sequence risk feature before PPO/walk-forward training.",
    ],
}

(CARD / "selection.json").write_text(json.dumps(selection, indent=2, ensure_ascii=True), encoding="utf-8")
(CARD / "summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=True), encoding="utf-8")

print(json.dumps(selection, indent=2))

{
  "experiment": "deep_cap_failure_attribution_v1",
  "created_at": "2026-05-25T16:05:36.233810+00:00",
  "source_experiment": "deep_capitulation_reentry_guard_v1",
  "direct_comparison_baseline": "oos_poor_window_factor_diff_aggregate_v1",
  "decision": "diagnostic_handoff_joint_participation_drawdown_no_promotion",
  "next_mechanism": "state_conditioned_participation_objective_with_explicit_drawdown_controller",
  "promotion_allowed": false,
  "full_gate": false,
  "promotion_blocker": "This is a diagnostic attribution notebook; deep-cap still fails the <=15% MaxDD objective and walk-forward gate.",
  "target_excess_pp": 30.0,
  "target_max_dd_pct": 15.0,
  "cluster_count": 3,
  "top_cluster": {
    "cluster_id": "C002",
    "cluster_start": "2025-04-04",
    "cluster_end": "2025-07-22",
    "profile_count": 6,
    "window_count": 37,
    "profiles": "balanced_rule_residual_v19_reg,balanced_rule_residual_v22,current_runtime_paper,deep_capitulation_reentry_guard_v1,drawdown_rebound_r

## Conclusion

The result of this notebook is intentionally a handoff, not a promotion. Deep-cap confirms that the model can add return when forced into rebound participation, but the hard objective is still not satisfied because MaxDD remains too high across strict OOS and walk-forward windows. The next valid loop should jointly optimize participation and drawdown rather than repeat threshold-only reentry grids or another imbalance-weighted BC screen.